# 1GabungBerita

In [ ]:
import gdown
files = {
    'detikv2_01.csv': '1nZeRV-dzOLKVMI8pUg8MCYq8natMYdnZ',
    'detikv2_02.csv': '1KAuERThGojW-M1ym5ZnNbxZuP_xMMHMY',
}

for name, file_id in files.items():
    print(f'Downloading {file_id} to {name}')
    gdown.download(id=file_id, output=name, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1nZeRV-dzOLKVMI8pUg8MCYq8natMYdnZ
To: /content/detikv2_01.csv
100%|██████████| 37.4M/37.4M [00:00<00:00, 54.4MB/s]


Downloading...
From: https://drive.google.com/uc?id=1KAuERThGojW-M1ym5ZnNbxZuP_xMMHMY
To: /content/detikv2_02.csv
100%|██████████| 38.3M/38.3M [00:01<00:00, 26.7MB/s]


In [ ]:
import pandas as pd

# 🔹 Daftar file CSV
files = [
    'detikv2_01.csv',
    'detikv2_02.csv',
]

# 🔹 Baca semua file, lewati baris rusak
df_list = [pd.read_csv(f, on_bad_lines='skip', encoding='utf-8') for f in files]
df = pd.concat(df_list, ignore_index=True)

# 🔹 Pastikan kolom tanggal valid
df['tanggal'] = pd.to_datetime(df['tanggal'], errors='coerce')

# 🔹 Hapus baris tanpa tanggal
df = df.dropna(subset=['tanggal'])

# 🔹 Urutkan berdasarkan tanggal
df = df.sort_values(by='tanggal', ascending=True).reset_index(drop=True)

# 🔹 Hapus kolom id lama jika ada
if 'id' in df.columns:
    df = df.drop(columns=['id'])

# 🔹 Tambahkan kolom id baru
df['id'] = range(1, len(df) + 1)

# 🔹 Pindahkan kolom id ke paling kiri
cols = ['id'] + [c for c in df.columns if c != 'id']
df = df[cols]
df.to_csv("detikv2_2025.csv", index=False)
print(df.head())

   id    tanggal                                              judul  \
0   1 2025-01-01  Video: Suara-suara Lega Warga Dengar PPN 12% C...   
1   2 2025-01-01  Batal Naik, Segini Tarif PPN Terbaru Di Minima...   
2   3 2025-01-01    Perjalanan Maju-Mundur Rencana Kenaikan PPN 12%   
3   4 2025-01-01  Ford Tarik 20.484 SUV Hybrid Karena Masalah Ba...   
4   5 2025-01-01  Daftar Lengkap Harga BBM Pertamina Terbaru di ...   

                                                 isi  
0  Sejumlah warga mengaku sempat khawatir dan gel...  
1  Jakarta - Pemerintah resmi mengumumkan kenaika...  
2  Jakarta - Kebijakan kenaikan Pajak Pertambahan...  
3  Jakarta - Administrasi Keselamatan Lalu Lintas...  
4  Jakarta - PT Pertamina (Persero) melakukan pen...  


In [ ]:
df

,id,tanggal,judul,isi
0,1,2024-01-01,McDonald's Gugat Gerakan Boikot Produk Israel ...,Jakarta - McDonald's Malaysia melayangkan guga...
1,2,2024-01-01,SKK Migas Sebut Target Produksi Minyak 1 Juta ...,Jakarta - Satuan Kerja Khusus Pelaksana Kegiat...
2,3,2024-01-01,Daging Shabu sampai Bakso Diskon 20% Setiap Ha...,Jakarta - Penuhi kebutuhan harian dengan belan...
3,4,2024-01-01,"Exit Tol KM 149 Ditutup, Begini Cara Pergi Ke ...",Jakarta - Akses pintu tol Gedebage Km 149 Jala...
4,5,2024-01-01,"Rayakan Tahun Baru, 5.670 Wisatawan Serbu Kepu...",Jakarta - Jumlah kunjungan wisatawan ke Kepula...
...,...,...,...,...
33006,33007,2024-12-31,"Nilai Transaksi Rp 14,9T, EPIC Sale Berhasil D...",Jakarta - Pemerintah meluncurkan program Every...
33007,33008,2024-12-31,"Video: Pertimbangkan Ini Sebelum Investasi, Te...",Berinvestasi bisa jadi opsi untuk menambah pen...
33008,33009,2024-12-31,"Investor Malaysia Bakal Masuk IKN Tahun Depan,...",Jakarta - Kepala Otorita Ibu Kota Nusantara (O...
33009,33010,2024-12-31,Prabowo Naikkan Harga Beras di Tingkat Petani ...,Jakarta - Pemerintahan Presiden Prabowo Subian...


# Spam

In [ ]:
import torch
import nltk
from nltk.tokenize import sent_tokenize
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# Pastikan modul nltk sudah siap
nltk.download('punkt')

# Muat model IndoBERT sentiment
MODEL_NAME = "mdhugol/indonesia-bert-sentiment-classification"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Label model (biasanya 0=neutral, 1=positive, 2=negative)
label_map = {0: "neutral", 1: "positive", 2: "negative"}

# Fungsi untuk analisis IndoBERT per berita (mirip VADER)
def indobert_sentiment(row, column):
    text = row[column]
    if not isinstance(text, str) or not text.strip():
        row['mean_score'] = 0.0
        row['mean_pos'] = 0.0
        row['mean_neg'] = 0.0
        row['mean_neu'] = 1.0
        row['per_sentence'] = []
        return row

    sentences = sent_tokenize(text)
    total_score = total_pos = total_neg = total_neu = 0.0
    per_sentence = []

    for i, sentence in enumerate(sentences, start=1):
        # Tokenisasi & prediksi
        inputs = tokenizer(sentence, return_tensors="pt", truncation=True)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0].numpy()

        # Ambil probabilitas per label
        prob_dict = dict(zip(label_map.values(), probs))

        # Hitung skor terarah seperti compound
        score = prob_dict["positive"] - prob_dict["negative"]

        total_score += score
        total_pos += prob_dict["positive"]
        total_neg += prob_dict["negative"]
        total_neu += prob_dict["neutral"]

        per_sentence.append({
            "sentence_ke": i,
            "sentence": sentence,
            "probs": prob_dict,
            "score": score
        })

    n = len(sentences)
    row["mean_score"] = total_score / n
    row["mean_pos"] = total_pos / n
    row["mean_neg"] = total_neg / n
    row["mean_neu"] = total_neu / n
    row["per_sentence"] = per_sentence

    return row


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# Contoh data
df = pd.DataFrame({
    "judul": [
        "Pemerintah dikritik atas kebijakan yang memberatkan masyarakat.",
        "Perusahaan energi terbarukan mencatat pertumbuhan positif tahun ini.",
        "IHSG stabil setelah pengumuman kebijakan baru."
    ]
})

# Gunakan tqdm agar ada progress bar
tqdm.pandas()
df = df.progress_apply(indobert_sentiment, axis=1, column="judul")

# Lihat hasil
print(df[["judul", "mean_score", "mean_pos", "mean_neg"]])


100%|██████████| 3/3 [00:00<00:00,  3.45it/s]

                                               judul  mean_score  mean_pos  \
0  Pemerintah dikritik atas kebijakan yang member...   -0.643729  0.177647   
1  Perusahaan energi terbarukan mencatat pertumbu...    0.927299  0.928002   
2     IHSG stabil setelah pengumuman kebijakan baru.    0.995957  0.996472   

   mean_neg  
0  0.821376  
1  0.000703  
2  0.000515  


In [ ]:
df

,judul,mean_score,mean_pos,mean_neg,mean_neu,per_sentence
0,Pemerintah dikritik atas kebijakan yang member...,-0.643729,0.177647,0.821376,0.000977,"[{'sentence_ke': 1, 'sentence': 'Pemerintah di..."
1,Perusahaan energi terbarukan mencatat pertumbu...,0.927299,0.928002,0.000703,0.071295,"[{'sentence_ke': 1, 'sentence': 'Perusahaan en..."
2,IHSG stabil setelah pengumuman kebijakan baru.,0.995957,0.996472,0.000515,0.003014,"[{'sentence_ke': 1, 'sentence': 'IHSG stabil s..."
